# 01 - Divisao treino/teste (train_test_split)

## Conceito
Avaliar um modelo no mesmo dado em que ele foi treinado gera uma estimativa
otimista e enganosa do desempenho. Precisamos separar os dados em:

- Treino: o modelo aprende os parametros aqui
- Teste: medimos o desempenho em dados nunca vistos

## Proporcoes tipicas
- 70/30 ou 80/20 para datasets medios
- 60/20/20 (treino/validacao/teste) quando ha ajuste de hiperparametros
- Com poucos dados: cross-validation e mais confiavel que holdout

## Por que estratificar
Em problemas de classificacao com classes desbalanceadas, um split aleatorio
pode deixar uma classe sub-representada no treino ou no teste. stratify=y
preserva a proporcao das classes em ambos os conjuntos.

## Por que embaralhar
Se os dados estao ordenados (por data, por id, por classe), um split sem
shuffle pode gerar conjuntos enviesados.

## Por que fixar random_state
Para reprodutibilidade. Sem isso, cada execucao gera um split diferente
e os resultados nao sao comparaveis.

In [12]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target          # type: ignore
feature_names = iris.feature_names     # type: ignore

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)
print("Classes:", np.unique(y))
print("Nomes das features:", feature_names)

Formato de X: (150, 4)
Formato de y: (150,)
Classes: [0 1 2]
Nomes das features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


In [13]:
# Split basico 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

print("Treino:", X_train.shape, y_train.shape)
print("Teste :", X_test.shape, y_test.shape)

Treino: (120, 4) (120,)
Teste : (30, 4) (30,)


## Estratificacao

Sem stratify, a proporcao das classes pode variar entre treino e teste.
Com stratify=y, ela e preservada.

In [14]:
from collections import Counter

# Sem estratificacao
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=1)
print("Sem stratify")
print("  treino:", Counter(y_tr))
print("  teste :", Counter(y_te))

# Com estratificacao
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)
print("\nCom stratify")
print("  treino:", Counter(y_tr))
print("  teste :", Counter(y_te))

Sem stratify
  treino: Counter({np.int64(2): 37, np.int64(0): 36, np.int64(1): 32})
  teste : Counter({np.int64(1): 18, np.int64(0): 14, np.int64(2): 13})

Com stratify
  treino: Counter({np.int64(0): 35, np.int64(2): 35, np.int64(1): 35})
  teste : Counter({np.int64(2): 15, np.int64(0): 15, np.int64(1): 15})


## Tres conjuntos: treino, validacao e teste

Quando vamos ajustar hiperparametros, precisamos de um conjunto de validacao
separado. Duas chamadas de train_test_split resolvem.

In [15]:
# Primeiro separa teste
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Depois separa validacao do restante
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print("Treino    :", X_train.shape)
print("Validacao :", X_val.shape)
print("Teste     :", X_test.shape)

Treino    : (90, 4)
Validacao : (30, 4)
Teste     : (30, 4)


## Cross-validation (KFold)

Alternativa mais robusta ao holdout. Divide os dados em K partes, treina K
vezes usando K-1 partes para treino e 1 para validacao, alternando qual parte
e a de validacao.

- Vantagem: usa todos os dados para validar em algum momento, resultado mais estavel
- Desvantagem: K vezes mais custoso
- StratifiedKFold preserva proporcao de classes em cada fold

In [16]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(max_iter=500)

# KFold estratificado com 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(modelo, X, y, cv=cv, scoring="accuracy")

print("Scores por fold:", scores.round(3))
print("Media:", scores.mean().round(3))
print("Desvio padrao:", scores.std().round(3))

Scores por fold: [1.    0.967 0.933 1.    0.933]
Media: 0.967
Desvio padrao: 0.03


## Exercicios

1. Faca um split 70/30 no dataset wine e compare as proporcoes de classes
   com e sem stratify.
2. Use cross_val_score com scoring="f1_macro" no dataset iris.
3. Compare a media de cross_val_score entre LogisticRegression e KNeighborsClassifier.
4. Explique com suas palavras por que nao devemos ajustar hiperparametros no conjunto de teste.